### MicroPython vs. Bare Metal C: Performance, Architecture, and Boundary Conditions

This notebook explores the architectural differences between CPython and MicroPython, evaluates the performance trade-offs between MicroPython and Bare Metal C on an STM32 microcontroller.

We will look at benchmarks involving GPIO, floating-point math, string formatting, array packing, and analyze the underlying source code.

---
### 1. CPython vs. MicroPython: Architectural Differences

While MicroPython implements the Python 3 syntax, its internal architecture is heavily optimized for resource-constrained environments (microcontrollers with limited RAM and Flash). 

**High-Level Differences:**
*   **Memory Management:** CPython uses reference counting backed by a garbage collector for cyclic references. MicroPython relies exclusively on a simple mark-and-sweep Garbage Collector (GC), avoiding the memory overhead of storing a reference count in every object.
*   **Compilation:** CPython parses code into an Abstract Syntax Tree (AST) and then compiles it to bytecode. MicroPython can compile directly from a parse tree to bytecode, skipping heavy AST generation to save RAM.
*   **Pre-compilation:** MicroPython as well allows code to be pre-compiled offline into `.mpy` files. As observed in our tests, a `4374` byte `.py` script shrinks to a `2249` byte `.mpy` bytecode file, saving valuable target storage and RAM during import.

**Low-Level Differences (Source Code Level):**
MicroPython uses a highly optimized object representation (`mp_obj_t`). Small integers and strings are often packed directly into the pointer itself (pointer tagging), saving heap allocations.

When it comes to floating-point operations, MicroPython minimizes VM overhead by directly encapsulating standard C library functions. This is why the performance gap between C and Python is smallest in pure math calculations. 

Take a look at how MicroPython defines math functions internally using C macros:

```c
#define MATH_FUN_2(py_name, c_name) \
    static mp_obj_t mp_math_##py_name(mp_obj_t x_obj, mp_obj_t y_obj) { \
        return math_generic_2(x_obj, y_obj, MICROPY_FLOAT_C_FUN(c_name)); \
    } \
    static MP_DEFINE_CONST_FUN_OBJ_2(mp_math_##py_name##_obj, mp_math_##py_name);

#if MICROPY_PY_MATH_ATAN2_FIX_INFNAN
mp_float_t atan2_func(mp_float_t x, mp_float_t y) {
    if (isinf(x) && isinf(y)) {
        return copysign(y < 0 ? MP_3_PI_4 : MP_PI_4, x);
    }
    return atan2(x, y);
}
MATH_FUN_2(atan2, atan2_func)
#else
MATH_FUN_2(atan2, atan2)
#endif
```

As seen above, MicroPython's `math.atan2` is just a thin wrapper (`MATH_FUN_2(atan2, atan2)`) around the hardware-optimized C `atan2` function[cite: 1]. The only overhead is popping the Python objects (`mp_obj_t`) off the VM stack, converting them to C floats, calling the native C math function, and boxing the result back into a Python object. 

### 2. Benchmark Source Code: Bare Metal C vs MicroPython

To compare performance, we execute identical logic on the STM32G0 in both environments. 

Here is the Bare Metal C implementation targeting STM32 HAL:
```c
/* Benchmark snippets from main.c */
uint32_t bench_gpio_toggle () {
	HAL_TIM_Base_Start(&htim2);
	uint32_t start_gpio = TIM2->CNT;
	for (volatile int i = 0; i < 100000; i++) {
		GPIOA->BSRR = TEST_PIN_Pin; // Set PA0 high
		GPIOA->BRR = TEST_PIN_Pin;  // Set PA0 low 
	}
	uint32_t end_gpio = TIM2->CNT;
	HAL_TIM_Base_Stop(&htim2);
	return (end_gpio - start_gpio);
}

uint32_t bench_floating_math () {
	volatile float ax = 0.5f, ay = -0.2f, az = 0.866f;
	volatile float roll, pitch;

	HAL_TIM_Base_Start(&htim2);
	uint32_t start_math = TIM2->CNT;
	for (int i = 0; i < 1000; i++) {
	    roll = atan2f(ay, az);
	    pitch = atan2f(-ax, sqrtf(ay * ay + az * az));
	}
	uint32_t end_math = TIM2->CNT;
	HAL_TIM_Base_Stop(&htim2);
	return (end_math - start_math);
}

uint32_t bench_string_formatting () {
	float voltages[16] = {3.2f, 3.3f, 3.4f, 3.5f, 3.6f, 3.7f, 3.8f, 3.9f, 4.0f, 4.1f, 4.2f, 3.1f, 3.2f, 3.3f, 3.4f, 3.5f};
	char bms_buffer[128];

	HAL_TIM_Base_Start(&htim2);
	uint32_t start_fmt = TIM2->CNT;
	for (volatile int i = 0; i < 5000; i++) {
		volatile int offset = 0;
	    for (volatile int j = 0; j < 16; j++) {
	        offset += snprintf(bms_buffer + offset, sizeof(bms_buffer) - offset, "%.2f,", voltages[j]);
	    }
	}
	uint32_t end_fmt = TIM2->CNT;
	HAL_TIM_Base_Stop(&htim2);
	return (end_fmt - start_fmt);
}

uint32_t bench_struck_pack () {
	uint8_t pack_buffer[16];
	uint32_t data_to_pack[4] = {100, 200, 300, 400};

	HAL_TIM_Base_Start(&htim2);
	uint32_t start_pack = TIM2->CNT;
	for (volatile int i = 0; i < 5000; i++) {
	    memcpy(pack_buffer, data_to_pack, sizeof(data_to_pack));
	}
	uint32_t end_pack = TIM2->CNT;
	HAL_TIM_Base_Stop(&htim2);
	return (end_pack - start_pack);
}
```

And here is the equivalent MicroPython benchmark script. Note the inclusion of GC management to ensure clean heap states before tests:

Note2: I wasn't able to use @native and @viper optimizations since Cortex M0 is ARMv6-M (Thumb-1) architecture which doesn't support ARMv7-M architecture instruction set - Thumb-2.

See: https://developer.arm.com/documentation/ddi0484/c/Introduction/Product-documentation--design-flow-and-architecture/Architecture-and-protocol-information?lang=en
which states: "The processor [context: Cortex M0/M0+] implements the ARMv6-M architecture profile..."

```python
import machine
import time
import gc
import math
import struct

def test_gpio_speed():
    try:
        pin = machine.Pin('A0', machine.Pin.OUT)
    except ValueError:
        return
    gc.collect()
    N = 100000
    pin.off()
    t0 = time.ticks_us()
    for _ in range(N):
        pin.on()
        pin.off()
    t1 = time.ticks_us()
    print(f"Time taken: {time.ticks_diff(t1, t0)} us")

def test_math_kinematics():
    gc.collect()
    N = 1000
    ax, ay, az = 0.5, -0.2, 0.866 
    t0 = time.ticks_us()
    for _ in range(N):
        roll = math.atan2(ay, az)
        pitch = math.atan2(-ax, math.sqrt(ay*ay + az*az))
        yaw = 0.1 * 0.01 
    t1 = time.ticks_us()
    print(f"Time taken: {time.ticks_diff(t1, t0)} us")

def test_memory_and_formatting():
    gc.collect()
    N = 5000
    dummy_voltages = [3.2, 3.5, 4.1, 3.8] * 4 
    t0 = time.ticks_us()
    for _ in range(N):
        log_str = ",".join(["{:.2f}".format(v) for v in dummy_voltages])
    t1 = time.ticks_us()
    print(f"Time taken: {time.ticks_diff(t1, t0)} us")

def test_struct_packing():
    gc.collect()
    N = 5000
    buf = bytearray(16)
    t0 = time.ticks_us()
    for i in range(N):
        struct.pack_into("<IIII", buf, 0, i, i+1, i+2, i+3)
    t1 = time.ticks_us()
    print(f"Time taken: {time.ticks_diff(t1, t0)} us")
```


### 3. Footprint & Speed Visualization

We can map the benchmark outputs from both firmware implementations using `matplotlib`. MicroPython adds significant base firmware size: the compiled C executable is roughly 42.3 KB, while the MicroPython firmware block occupies about 296.4 KB.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Data based on benchmark outputs (Execution times in microseconds)
c_speed = {"GPIO": 34419, "Math": 191080, "String": 10839986, "Struct": 16192}
py_speed = {"GPIO": 3525397, "Math": 332681, "String": 24852289, "Struct": 140329}
mpy_speed = {"GPIO": 3525400, "Math": 332698, "String": 24851257, "Struct": 140329}

py_gpio_variants = {
    "Regular (machine.Pin)": 3525398,
    "Direct Reg Access": 5532059,
    "Optimized Direct Reg Access": 2607305,
}
# 1. Size: mpy vs py
plt.figure(figsize=(8, 6))
plt.bar(["Source (.py)", "Compiled (.mpy)"], [4374, 2249], color=["#e74c3c", "#3498db"])
plt.title("Script Size: .py vs .mpy")
plt.ylabel("Bytes")
for i, v in enumerate([4374, 2249]):
    plt.text(i, v + 100, f"{v} B", ha="center")
plt.show()

# 2. Size: py vs C (Firmware footprint)
plt.figure(figsize=(8, 6))
plt.bar(["Bare Metal C", "MicroPython Base"], [42.3, 296.4], color=["#2ecc71", "#9b59b6"])
plt.title("Firmware Base Size: C vs MicroPython")
plt.ylabel("Kilobytes (KB)")
for i, v in enumerate([42.3, 296.4]):
    plt.text(i, v + 5, f"{v} KB", ha="center")
plt.show()


labels = list(c_speed.keys())
x = np.arange(len(labels))
width = 0.35
# 3. Speed: py vs C
fig, ax = plt.subplots(figsize=(8, 6))
ax.bar(x - width / 2, list(c_speed.values()), width, label="C", color="#2ecc71")
ax.bar(x + width / 2, list(py_speed.values()), width, label="Python", color="#e74c3c")

ax.set_ylabel("Execution Time (us) - Log Scale")
ax.set_title("Execution Speed: C vs Python")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_yscale("log")
ax.legend()
plt.show()

# 4. Size: GPIO optimizations
plt.figure(figsize=(8, 6))
labels_gpio = list(py_gpio_variants.keys())
values_gpio = list(py_gpio_variants.values())
bars = plt.bar(labels_gpio, values_gpio, color=["#e67e22", "#c0392b", "#d35400"])
plt.title("Python GPIO Access Strategies (Time in us)")
plt.ylabel("Execution Time (us)")
plt.xticks(rotation=15)
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, yval + 50000, f"{int(yval)} us", ha="center")
plt.tight_layout()
plt.show()

### 4. Analysis & Boundary Conditions

By analyzing the data, we can define the edge cases and applicability limits of MicroPython versus transitioning to C:

**1. Code Size & Memory Boundaries:**
MicroPython is out of the question if your microcontroller has less than 256KB of Flash and 16KB of RAM. The bare metal C firmware requires roughly 42.3 KB of space (including expensive `sprintf` usage), whereas the base MicroPython firmware requires 296.4 KB before you even write a line of Python. If cost scales strongly with Flash size in your production run, C is mandatory. 

**2. Computation Boundaries:**
*   **Math:** Surprisingly, math operations represent the *smallest* performance gap. Our tests showed C took ~191,080 us while MicroPython took ~332,681 us for 1000 RPY calculations. Because MicroPython passes floats directly to hardware FPU-backed C functions (as shown in the source code analysis), you don't lose much speed here.
*   **Data Packing / I/O:** `struct.pack` in MicroPython (512,260 us) is massively slower than a C `memcpy` (16,192 us). If your device relies on high-frequency sensor ingestion via SPI/I2C and immediate buffer packing for DMA transmission, MicroPython will choke.
*   **String Formatting:** MicroPython takes roughly 24.8 million us compared to C's 10.8 million us for 5000 string format iterations. Furthermore, MicroPython's string operations constantly fragment the heap, triggering Garbage Collection pauses that break real-time requirements.

**3. GPIO and Bit-Banging Boundaries:**
Bare metal C takes roughly 34,419 us to toggle GPIO 100,000 times. MicroPython using standard `machine.Pin` takes 3,525,398 us—over 100x slower. Even with highly optimized direct memory access via `machine.mem32`, MicroPython takes 2,607,305 us. MicroPython is therefore completely unsuitable for software-defined high-speed protocols (like bit-banging precise WS2812 LED timings or fast custom serial buses) unless utilizing dedicated PIO hardware (like on the RP2040).

**Conclusion: When to write in C vs MicroPython?**
Use **MicroPython** when time-to-market is critical, the logic is highly state-machine driven (like web connectivity, MQTT, or UI rendering), and hardware interactions are low-frequency or handled by asynchronous DMA/interrupts. The overhead of writing and maintaining UI/Networking code in C is massive compared to Python.

Switch to **C** when strict deterministic real-time execution is required, when the unit economics of the hardware demand smaller Flash/RAM sizes, or when the application is fundamentally bound by high-frequency GPIO polling or heavy byte-buffer manipulations.

### 5. Repeat results of benchmarks
Here is comprehensive way to replicate benchmarks i use. Specific for STM32G0B1 MINI-NODE-V3.

In [ ]:
# From micropython/ports/stm32 build interpreter for MINI_NODE_V3
! make BOARD=MINI_NODE_V3 -j$(nproc)

# If build finished too fast or something is wrong try cleaning first (uncomment when need)
# ! make BOARD=MINI_NODE_V3 clean

To Flash MINI_NODE_V3:
   1. Open STM32CubeProgrammer.
   2. Press "+" $\rightarrow$ "Open file" in the "Memory & file editing" section.
   3. Choose: ports/stm32/build-MINI_NODE_V3/firmware.elf
   4. Press "Connect" (if an error occurs, check connectivity on SWD pins).
   5. Press "Download" (if an error occurs, press Download again).
   6. STM32CubeProgrammer should flash your STM now.

In [ ]:
# Now we can try to talk to our MCU.
# Ensure that USART1 is connected to your USB-UART adapter!
# Use any of the /dev/ttyUSB* tool to open port (i use tio):
! tio /dev/ttyUSB* -b 115200
# If connection is successful - press enter and REPL's ">>>" should appear
# If nothing happened - try to repower MINI_NODE_V3. Note that most reliable way to tell is micropython is running on MINI_NODE_V3 is look at led.
# If white then micropython is okay - better look in to UART connectivity first then.

In [ ]:
# Now leave the tio - we ensured that we have connection with our node
# Precompile .mpy file
! ../mpy-cross/build/mpy-cross -O3 -march=armv6m benches_adv.py

In [ ]:
# Copy files into node:
! mpremote connect /dev/ttyUSB* cp benchmark/benches_adv.mpy :benchmark_compiled.mpy
! mpremote connect /dev/ttyUSB0 cp benchmark/benches_adv.py :benchmark.py  

In [ ]:
# Return to tio and have fun with benchmark functions. You may take peek into `full_terminal_output.txt` to check one of ways of interaction
! tio /dev/ttyUSB* -b 115200